# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第8章：ベイジアンマルチレベルモデル：brmsとStanによる推定


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Vasishth, Nicenboim, Beckman, Li & Kong (2018), *Journal of Phonetics*／Bürkner (2017), *Journal of Statistical Software*。

In [ ]:
# ⏳ brmsのインストールには数分かかることがある（Stanという別言語の
# コンパイラ一式が付随するため、mgcv等より重い）。
install.packages(c("brms", "bayesplot"), repos = "https://cloud.r-project.org")
library(brms)       # ベイズマルチレベルモデル（内部でStanを呼ぶ）
library(bayesplot)  # 事後分布の可視化


In [ ]:
# ---- 共通データセットの読み込み ----
# 第1章と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、分散推定値がすべて0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7章のAR(1)を使うときだけ例外）。


■ lmer()からbrm()へ：

**実際の研究例**：Vasishth, Nicenboim, Beckman, Li & Kong (2018, *Journal of Phonetics*) のチュートリアルは、VOTデータを題材に頻度論の混合効果モデルとベイズのマルチレベルモデルを並べて示した。頻度論のp値は「効果が0だとしたらこのデータが観測される確率」を返すが、研究者が本当に知りたいのは多くの場合「効果が正である確率はどれくらいか」である。ベイズ推定は事後分布としてこれに直接答える。

第5章と同じモデル構造をbrmsで書く。構文はlme4とほぼ同じだが、結果は事後分布として得られる（Bürkner, 2017; Vasishth et al., 2018）。

初回のフィットはStanのコンパイルが走るため数分かかる。

In [ ]:
# ⏳ このセルは実行に1〜3分ほどかかる。初回はまずStanのCコードを
# コンパイルする（"Compiling Stan program..." と表示される）。その後、
# 4本の鎖(chain)がそれぞれ2000回サンプリングする。"Chain 1: Iteration..."
# のような進捗表示が出ていれば正常に動いている。
model_brm <- brm(
  VOT ~ condition + (1 + condition | speaker) + (1 | word),
  data = vot_data,
  family = gaussian(),        # 応答が連続量なので正規分布を仮定
  chains = 4, iter = 2000, warmup = 1000,  # MCMCを4本の独立した鎖で回し、各2000反復のうち最初1000を捨てる
  seed = 42
)

summary(model_brm)
# lmer()の"Estimate"に相当するのがEstimate列、
# 信用区間(l-95% CI, u-95% CI)がp値の代わりに解釈の中心になる。

# --- 実行結果（抜粋）---
# Regression Coefficients:
#                    Estimate Est.Error l-95% CI u-95% CI Rhat
# Intercept             22.82      2.15    18.51    27.01 1.00
# conditionvoiceless    52.23      6.80    38.48    65.89 1.00
#
# 警告: There were 1 divergent transitions after warmup.
#
# 【出力の読み方】
# ・第5章のlmer()と近い推定値が得られている（切片22.82、条件の効果52.23）。
#   事前分布も収束診断もまだ扱っていないこの時点では、divergent transition
#   の警告が1件出る。対処は8.3節・8.4節で扱う。


■ 事後分布の可視化：

In [ ]:
mcmc_plot(model_brm, variable = "b_conditionvoiceless", type = "hist")  # 1つの係数の事後分布
mcmc_plot(model_brm, type = "areas", regex_pars = "^b_")  # 固定効果の係数をまとめて表示


■ 事後分布から確率を直接読む：

事後分布は「モデルの仮定のもとで、データを見たあとのパラメータについての確率分布」である。MCMCで取り出した事後サンプルのうち、ある命題が成り立つものの割合が、そのままその命題の事後確率になる。上のモデルで、条件の効果が0より大きい確率、40 msを超える確率、45〜60 msにある確率を数えて確かめる。

In [ ]:
# model_brm は上のセルで当てはめたものをそのまま使う（同じコード・seed = 42）。
# 事後サンプルを取り出し、条件の効果（b_conditionvoiceless）の列を使う
draws <- as_draws_df(model_brm)
beta  <- draws$b_conditionvoiceless
cat("事後サンプル数:", length(beta), "\n")

# 「条件が成り立つサンプルの割合」がそのまま事後確率になる
cat("(a) 効果が0より大きい確率      :", mean(beta > 0), "\n")
cat("(b) 効果が40 msを超える確率    :", round(mean(beta > 40), 3), "\n")
cat("(c) 効果が45〜60 msにある確率  :", round(mean(beta > 45 & beta < 60), 3), "\n")
cat("(d) 事後中央値                 :", round(median(beta), 2), "\n")
ci <- quantile(beta, c(0.025, 0.975))
cat("    95%信用区間                : [", round(ci[1], 2), ",", round(ci[2], 2), "]\n")

# brmsの hypothesis() も同じ計算（(b)と照合）。Estimateは「効果 − 40」の事後平均
hypothesis(model_brm, "conditionvoiceless > 40")$hypothesis[, c("Estimate", "Post.Prob")]

# --- 実行結果 ---
# 事後サンプル数: 4000
# (a) 効果が0より大きい確率      : 1
# (b) 効果が40 msを超える確率    : 0.964
# (c) 効果が45〜60 msにある確率  : 0.756
# (d) 事後中央値                 : 52.21
#     95%信用区間                : [ 38.48 , 65.89 ]
#   Estimate Post.Prob
# 1 12.23414   0.96375
#
# 【出力の読み方】
# ・95%信用区間（38.48〜65.89）は上の summary() の l-95% CI・u-95% CI と
#   同じ値である。summary() もこの4000個のサンプルの分位点を報告している。
# ・(a) が1なのは、4000個すべてが0より大きかったということ。「確率がちょうど1」
#   という断言ではなく、このサンプル数では0以下の値が出なかったと読む。
# ・(b) 0.964、(c) 0.756のように、問いの形を変えるだけで同じ事後サンプルから
#   別の確率が得られる。hypothesis() の Post.Prob（0.96375）は (b) と一致する。
# ★これらの確率はモデルと事前分布が正しいという仮定に条件づいている。
#   確率を報告するときは、事前分布と収束診断（8.4節）を合わせて示す。


■ 弱情報事前分布の指定：

Vasishth et al. (2018) が推奨する、経験的に妥当な範囲に緩く制約するweakly informative priorの例。**ただし「弱情報」かどうかは、数式の見た目ではなく、その分布が示唆する値の範囲がデータの尺度に対して妥当かどうかで決まる。** `exponential(1)`のような一見無難に見える既定値も、応答変数の尺度（本書ではVOT、ミリ秒単位で数十の桁）に対して見合っていなければ、実質的に強い制約になりうる。まず**事前予測分布**を確認してから、モデルに組み込む。

In [ ]:
# ---- 事前予測チェック：class="sd"（ランダム効果の標準偏差）の事前分布 ----
# データをまだ見ずに、事前分布そのものがどんな値を「ありそう」だと
# 言っているかを確認する。exponential(rate)の平均は 1/rate である。
cat("exponential(1) の事前分布（平均1ms）が示唆する範囲:\n")
print(round(quantile(rexp(20000, rate = 1), c(0.1, 0.5, 0.9)), 2))

cat("\nexponential(0.1) の事前分布（平均10ms）が示唆する範囲:\n")
print(round(quantile(rexp(20000, rate = 0.1), c(0.1, 0.5, 0.9)), 2))

# --- 実行結果 ---
# exponential(1) の事前分布（平均1ms）が示唆する範囲:
#  10%  50%  90%
# 0.11 0.69 2.34
#
# exponential(0.1) の事前分布（平均10ms）が示唆する範囲:
#   10%   50%   90%
#  1.01  6.77 22.76
#
# 【出力の読み方】
# ・**exponential(1)は、90%の確率で標準偏差が約2.3ms未満だと言っている（乱数による近似出力は2.34）。**
#   しかし本書のVOTデータでは、話者間の条件効果のばらつき（ランダム傾きの
#   標準偏差）は生成時点で10msに設定してある。つまりこの「弱情報」のつもりの
#   事前分布は、真の値の妥当な範囲をほとんど除外してしまっている——
#   これは弱情報ではなく、かなり強い制約である。
# ・exponential(0.1)（平均10ms）なら、10msという値は分布の裾ではなく本体
#   （50%点が6.77ms、90%点が22.76ms）に収まり、VOTの尺度に対してもっともらしい。
#   ★指数分布の密度は0で最大となり単調減少するので、最頻値（モード）は常に0
#   である点に注意——「10msが分布の中心」という言い方は、正規分布のような
#   対称分布には正しいが、指数分布には厳密には当てはまらない。ここで言いたいのは
#   「10msが極端に起こりにくい値ではない」ということである。
#   **事前分布の「強さ」は係数の値ではなく、データの尺度と比べて初めて判断できる。**
#   `class = "sd"`はモデルが推定する標準偏差パラメータであり、分散
#   （標準偏差の2乗）ではない点にも注意する——コメントで「分散パラメータ」と
#   書くと式と食い違う。


In [ ]:
priors_old <- c(
  prior(normal(50, 30), class = "Intercept"),
  prior(normal(0, 30), class = "b"),
  prior(exponential(1), class = "sd")     # ★平均1ms。事前予測チェックで見たとおり強すぎる
)
priors_new <- c(
  prior(normal(50, 30), class = "Intercept"),
  prior(normal(0, 30), class = "b"),
  prior(exponential(0.1), class = "sd")   # VOTの尺度に合わせ、平均10msに校正
)

# ⏳ 事前分布が変わるとStanのコードも再コンパイルされるため、これも1〜3分かかる。
# 2つの事前分布で同じモデルを当て、感度分析として結果を比べる。
model_brm_old   <- brm(VOT ~ condition + (1 + condition | speaker) + (1 | word),
                        data = vot_data, prior = priors_old,
                        chains = 4, iter = 2000, warmup = 1000, seed = 42,
                        control = list(adapt_delta = 0.95))
model_brm_prior <- brm(VOT ~ condition + (1 + condition | speaker) + (1 | word),
                        data = vot_data, prior = priors_new,
                        chains = 4, iter = 2000, warmup = 1000, seed = 42,
                        control = list(adapt_delta = 0.95))

vc_old <- as.data.frame(VarCorr(model_brm_old)$speaker$sd)["conditionvoiceless", ]
vc_new <- as.data.frame(VarCorr(model_brm_prior)$speaker$sd)["conditionvoiceless", ]
cat("話者間・条件効果のSD、真値は10:\n")
cat("  exponential(1) （強すぎる事前分布）:", round(vc_old$Estimate, 2),
    "[", round(vc_old$Q2.5, 2), ",", round(vc_old$Q97.5, 2), "]\n")
cat("  exponential(0.1)（校正した事前分布）:", round(vc_new$Estimate, 2),
    "[", round(vc_new$Q2.5, 2), ",", round(vc_new$Q97.5, 2), "]\n")

summary(model_brm_prior)

# --- 実行結果 ---
# 話者間・条件効果のSD、真値は10:
#   exponential(1) （強すぎる事前分布）: 11.43 [ 8.47 , 15.31 ]
#   exponential(0.1)（校正した事前分布）: 17.94 [ 11.47 , 28.76 ]
#
# 【出力の読み方】
# ・このデータ・この乱数seedでは、exponential(1)の点推定（11.43）のほうが
#   校正後（17.94）より真値の10に近い——**だからといって強すぎる事前分布で
#   問題ないと結論してはいけない。** exponential(1)の信用区間は8.47〜15.31と
#   狭く、これは「データがそれだけ強く語っている」からではなく、**事前分布
#   自身が狭い範囲しか許していない**ために起きている。狭い区間が正しさの
#   証拠になるとは限らない。
# ・校正後の事前分布は信用区間が11.47〜28.76と大きく広がり、真の不確実性を
#   より正直に映している。話者10名・1話者20語というこの規模のデータでは、
#   ランダム効果の分散成分をピンポイントで特定するだけの情報量が本来無い
#   ——事前分布を締め付けて区間を狭く見せることは、その情報の乏しさを
#   隠すことにしかならない。
# ・**弱情報事前分布を使う目的は、係数を都合よく動かすことではなく、
#   常識的にありえない範囲（負の分散、数千msのVOTなど）を除外しつつ、
#   データに語らせる余地を残すことである。** 事前予測チェックを飛ばして
#   「弱情報のつもり」の事前分布を使うと、この目的を裏切ることがある。


■ 事前分布の幅 × データ量：事前分布はどれだけ結論を動かすか：

事前分布の影響の大きさは、事前分布の幅だけでなくデータの情報量にも左右される。話者数（3・5・10名）と条件の効果の事前分布の幅（normal(0, 2)・normal(0, 10)・normal(0, 30)）を組み合わせた9通りの当てはめで確かめる（真の効果は52.5 ms）。

In [ ]:
# ⏳ 9回のモデル当てはめ（各回でStanのコンパイルを含む）で10〜20分ほどかかる。
res <- list()
for (n_sp in c(3, 5, 10)) {
  sp <- levels(factor(vot_data$speaker))[1:n_sp]
  d <- droplevels(subset(vot_data, speaker %in% sp))
  for (s in c(2, 10, 30)) {
    f <- brm(VOT ~ condition + (1 + condition | speaker) + (1 | word), data = d,
      prior = c(set_prior("normal(50, 30)", class = "Intercept"),
                set_prior(paste0("normal(0, ", s, ")"), class = "b"),
                set_prior("exponential(0.1)", class = "sd")),
      chains = 4, iter = 2000, warmup = 1000, seed = 42, refresh = 0,
      control = list(adapt_delta = 0.99, max_treedepth = 12))
    fe <- fixef(f)["conditionvoiceless", ]
    res[[length(res) + 1]] <- data.frame(n_speakers = n_sp, prior_sd = s,
      mean = round(fe["Estimate"], 2), sd = round(fe["Est.Error"], 2),
      lo = round(fe["Q2.5"], 1), hi = round(fe["Q97.5"], 1))
  }
}
print(do.call(rbind, res), row.names = FALSE)

# --- 実行結果 ---
#  n_speakers prior_sd  mean    sd   lo   hi
#           3        2  0.47  1.95 -3.4  4.3
#           3       10 11.48  9.96 -7.8 31.5
#           3       30 52.39 13.59 21.3 75.0
#           5        2  0.61  2.00 -3.3  4.5
#           5       10 17.83 10.74 -3.3 38.7
#           5       30 54.02  8.47 35.5 68.9
#          10        2  1.01  1.99 -3.0  5.0
#          10       10 31.80  9.63  9.8 47.4
#          10       30 50.54  5.76 38.4 61.4
#
# 【出力の読み方】
# ・normal(0, 2)ではデータを増やしても事後平均がほぼ動かない（0.47→0.61→1.01）。
#   事前分布が「効果はほぼ±4 ms」と強く主張しているため、10名に増やしても
#   真の効果52.5 msに近づかない。
# ・normal(0, 10)では事前分布の影響が話者数とともに薄れる（11.48→17.83→31.80）が、
#   10名でも真の値には届かない。
# ・normal(0, 30)は3名でも真の値に近い推定を返し、事後SDが話者数とともに縮む
#   （13.59→8.47→5.76）。事前分布が十分に緩ければ、データ量は不確実性の小ささとして現れる。
# ★事前分布の幅そのものは標本サイズと無関係だが、事後分布をどれだけ引っぱるかは
#   データの情報量との相対で決まる。自分のデータ量で結論が動かないかを感度分析で確認する。


■ 収束の診断（結果を読む前に必ず見る）：

MCMCは「事後分布からサンプルを取ってくる」手続きなので、**サンプリングが失敗していれば推定値そのものが信用できない**。係数を読む前に、次の3つを必ず確認する。

1. **Rhat**（各パラメータ）… 4本の鎖が同じ分布に落ち着いたかの指標。1.01より大きい値があれば、まだ収束していないと考えてiterを増やす。
2. **ESS（有効サンプルサイズ）** … 自己相関を考慮した実質的なサンプル数。Bulk_ESS・Tail_ESSとも400を下回るなら、区間推定の端が不安定になる。
3. **divergent transitions** … 警告が出た場合は`adapt_delta`を上げる（上のセルで0.95にしてある）。少数なら実害が無いこともあるが、**警告を無視して結果だけ報告してはいけない**。

In [ ]:
# RhatとESSを一覧する。max(Rhat)が1.01未満、min(ESS)が400以上なら一応合格。
post_summary <- posterior::summarise_draws(model_brm_prior)
cat("max Rhat      :", round(max(post_summary$rhat, na.rm = TRUE), 4), "\n")
cat("min Bulk_ESS  :", round(min(post_summary$ess_bulk, na.rm = TRUE)), "\n")
cat("min Tail_ESS  :", round(min(post_summary$ess_tail, na.rm = TRUE)), "\n")

# 発散遷移（divergent transitions）が何回起きたかを数える
np <- brms::nuts_params(model_brm_prior)
cat("divergences   :", sum(subset(np, Parameter == "divergent__")$Value), "\n")

# 鎖ごとの軌跡。4本が同じ帯の中で混ざり合っていれば収束している（"毛虫"の形）。
mcmc_plot(model_brm_prior, type = "trace", regex_pars = "^b_")


■ 事後予測チェックとLOO：

In [ ]:
pp_check(model_brm_prior, ndraws = 100)
# モデルからシミュレーションしたデータ（薄い線）と実データ（濃い線）の分布を重ねて描く。
# 大きくずれていれば、モデルの仮定（正規分布など）が実データに合っていない疑い。

loo(model_brm_prior)
# leave-one-out交差検証によるモデル比較の指標。複数モデルを比べるときに使う。


■ モデル比較：LOOでランダム傾きの要否を確かめる：

ランダム傾きを持つモデルと持たないモデルを実際にLOOで比べ、第5章で確認した「話者ごとに条件の効果が違う」という事実と整合するかを見る。

In [ ]:
# ⏳ 数十秒。ランダム傾きを持たないモデルを追加でfitする。
model_brm_noslope <- brm(VOT ~ condition + (1 | speaker) + (1 | word),
                          data = vot_data, prior = priors_new,
                          chains = 4, iter = 2000, warmup = 1000, seed = 42,
                          control = list(adapt_delta = 0.95))

loo_full <- loo(model_brm_prior)
loo_noslope <- loo(model_brm_noslope)
loo_compare(loo_full, loo_noslope)

# --- 実行結果 ---
#                     elpd_diff se_diff
# model_brm_prior       0.0       0.0
# model_brm_noslope  -100.9      10.7
#
# 【出力の読み方】
# ・elpd_diffは標本外予測性能（大きいほど良い）の差。ランダム傾きを持つ
#   モデルを基準（0.0）とすると、持たないモデルは100.9小さく、標準誤差
#   （10.7）の約9.4倍にあたる——差が標準誤差の2〜3倍を大きく超えており、
#   ランダム傾きを持つモデルのほうが明確に予測性能が高いと判断できる。
# ・これは第5章で確認した「条件の効果の話者差（傾きの標準偏差16.68ms）が
#   無視できない大きさだった」という事実と整合する。
# ・LOOが答えるのは「どちらが将来の観測をより正確に予測するか」であって、
#   「分散はゼロと区別できるか」という検定の問いではない。ランダム効果に
#   何を入れるかは検定ではなく実験デザインから決める（第16章16.2節）のが
#   本書の立場であり、LOOはその判断を事後的に確認する道具として使う。


■ 演習：

1. `P(β_condition > 0 | data)`を事後サンプルから直接計算せよ（`hypothesis(model_brm_prior, "conditionvoiceless > 0")`）。
2. 事前分布をより強い（狭い）ものに変え、事後分布への影響をsensitivity analysisとして確認せよ。